In [5]:
import re
import numpy as np


def extract_entity_features(token, position):
    """
    Extract simple features for a token to decide whether it may be
    part of a Named Entity.
    """
    is_first_word = position == 0

    x1 = 1 if token[0].isupper() else 0
    x2 = 1 if token.isupper() else 0
    x3 = 1 if any(char.isdigit() for char in token) else 0
    x4 = 1 if is_first_word else 0
    x5 = len(token)

    return np.array([x1, x2, x3, x4, x5])


def extract_named_entity_candidates(text):
    """
    Extract possible Named Entities by selecting words that start with
    uppercase letters, skipping the first word of each sentence, and
    grouping consecutive entity-like tokens into n-grams.

    Examples:
    - La Odisea
    - Italia
    - Ciudad de Nueva York
    """
    connector_words = {"de", "del", "la", "las", "los", "y", "en"}

    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    entities = []

    for sentence in sentences:
        tokens = re.findall(r"\b\w+\b", sentence)

        current_entity = []

        for i, token in enumerate(tokens):
            features = extract_entity_features(token, i)

            starts_with_uppercase = features[0] == 1
            is_first_word = features[3] == 1
            is_connector = token.lower() in connector_words

            if starts_with_uppercase and not is_first_word:
                current_entity.append(token)

            elif is_connector and current_entity:
                current_entity.append(token)

            else:
                if current_entity:
                    # Avoid ending an entity with a connector
                    while current_entity and current_entity[-1].lower() in connector_words:
                        current_entity.pop()

                    if current_entity:
                        entities.append(" ".join(current_entity))

                    current_entity = []

        if current_entity:
            while current_entity and current_entity[-1].lower() in connector_words:
                current_entity.pop()

            if current_entity:
                entities.append(" ".join(current_entity))

    return entities


text = """
  Homero escribió La Odisea.
  El poema menciona a Ulises y a Penélope.
  Viajé a Italia y después a Ciudad de Nueva York.
  La Universidad Nacional Autónoma está en México.
"""

entities = extract_named_entity_candidates(text)

print(entities)

['La Odisea', 'Ulises', 'Penélope', 'Italia', 'Ciudad de Nueva York', 'Universidad Nacional Autónoma', 'México']
